##Download and import necessary library

In [1]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.6 MB/s eta 0:00:00


In [2]:
from sklearn.metrics import confusion_matrix, f1_score
import seaborn as sns
import matplotlib.pyplot as plt
from ultralytics import YOLO
import cv2
import os
import random
import shutil

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


##1- Download and locate data in drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')
data="drive/MyDrive/data/data1.yaml"

Mounted at /content/drive


##2- Preprocess :


1.   Changing the name of the image because the process of downloading images from Google using the **" [Image Downloader - Save pictures](https://chromewebstore.google.com/detail/image-downloader-save-pic/daeljdgmllhgmbdkpgnaojldjkdgkbjg) "**  tool leads to similarity in the names of the images, and when we repeat the process on different categories, it leads to override of the image.

1.    Deleting very poor quality and not relating images about our classes.

2.   Resizing images to make all images same size and input it to model .








In [ ]:
source_directory = 'Mahashi'
destination_directory = source_directory+'_change'

if not os.path.exists(destination_directory):
    os.makedirs(destination_directory)

file_list = sorted(os.listdir(source_directory))

number = 0

for filename in file_list:
    current_file_path = os.path.join(source_directory, filename)

    if not filename.lower().endswith(('.txt',)):
        new_filename =source_directory+ f"_{number}.jpg"
        number += 1

    new_file_path = os.path.join(destination_directory, new_filename)

    shutil.copy2(current_file_path, new_file_path)
    print(f"Copied and renamed: {filename} -> {new_filename}")

print("Copying and renaming completed.")

input_folder = destination_directory
output_folder = destination_directory+"_resize"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
image_files = [f for f in os.listdir(input_folder) if f.endswith(('.jpeg', '.jpg', '.png', '.gif', '.bmp', '.tiff', '.tif', '.webp', '.heic', '.heif', '.cr2', '.cr3', '.nef', '.nrw', '.arw', '.raf', '.orf', '.rw2', '.dng', '.svg', '.psd','.ico'))]
new_size = (960, 960)
counter=0
for image_file in image_files:
    img_path = os.path.join(input_folder, image_file)

    img = cv2.imread(img_path)
    if img is None:
        print(f"Error: Image Not Found {img_path}")
        continue
    img_resized = cv2.resize(img, new_size)
    output_path = os.path.join(output_folder, image_file)

    cv2.imwrite(output_path, img_resized)
    counter+=1

##3- Visualization of images with their class and a box around the object .

In [ ]:
data_directory = "/content/drive/MyDrive/data/food"
class_names = [
    "Pizza", "Mansaf", "Mjadra", "Maqluba", "Fasolia", "Awameh", "Salad",
    "Molokhia", "Shawarma", "Macaroni", "Burger", "Kunafa", "Dawali",
    "Kabsa", "Msakan", "Qatayif", "Cucumber with Yogurt", "Mahashi",
    "Coleslaw Salad", "Muhalabia"
]
num_images_to_visualize = 8

def Get_Label(annotations):
    if len(annotations) == 0:
        label = "None"
    else:
        label = class_names[int(annotations[0].split()[0])]
    return label

image_files = [f for f in os.listdir(data_directory) if f.lower().endswith(('.jpeg', '.jpg', '.png'))]

if not image_files:
    print(f"No image files found in {data_directory}")
else:
    images_to_process = random.sample(image_files, min(num_images_to_visualize, len(image_files)))
    for image_file in images_to_process:
        img_path = os.path.join(data_directory, image_file)
        txt_file = os.path.splitext(image_file)[0] + '.txt'
        txt_path = os.path.join(data_directory, txt_file)

        img = cv2.imread(img_path)
        if img is None:
            print(f"Error: Failed to load image {img_path}. Skipping.")
            continue

        annotations = []
        if os.path.exists(txt_path):
            with open(txt_path, 'r') as f:
                annotations = f.readlines()
        else:
            print(f"Warning: Annotation file not found for {image_file} at {txt_path}")

        img_height, img_width, _ = img.shape

        for annotation in annotations:
            parts = annotation.split()
            if len(parts) == 5:
                class_id, x_center, y_center, width, height = map(float, parts)
                x_center *= img_width
                y_center *= img_height
                width *= img_width
                height *= img_height
                x_min = int(x_center - width / 2)
                y_min = int(y_center - height / 2)
                x_max = int(x_center + width / 2)
                y_max = int(y_center + height / 2)
                cv2.rectangle(img, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)

        label = Get_Label(annotations)

        plt.figure(figsize=(7, 7))
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title(f"Image: {label}")
        plt.axis('off')
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

##4- Feature Engineering :


*  Apply [annotations](https://www.cvat.ai/) technique to images to insert them into the model with their txt file containing the class number and the 4 corners of the rectangle .
*    Involves annotating images with class labels and bounding box coordinates. This structured data, along with the class names and dataset paths defined in the YAML file, is crucial for training the object detection model. The annotations and YAML file together provide the necessary information for the model to learn object locations and identities.



##5- Modeling :



*   Splitting the data to training , testing and validation in yaml file to train the YOLO model .

*   YOLOv8s model is initialized and trained for 130 epochs using the defined dataset. The training process utilizes parameters like image size and learning rate, and the best-performing model is saved to Google Drive for later use.

*   To guarantee consistent and stable convergence, a learning rate of 0.001 was used. Smaller numbers severely hindered training, whereas larger values caused instability.
*   Save the model to use test and deploy it .





In [ ]:
data_yaml_content = """
names:
  - Pizza
  - Mansaf
  - Mjadra
  - Maqluba
  - Fasolia
  - Awameh
  - Salad
  - Molokhia
  - Shawarma
  - Macaroni
  - Burger
  - Kunafa
  - Dawali
  - Kabsa
  - Msakan
  - Qatayif
  - Cucumber with Yogurt
  - Mahashi
  - Coleslaw Salad
  - Muhalabia

train: /content/drive/MyDrive/data/train
val: /content/drive/MyDrive/data/val
test: /content/drive/MyDrive/data/test
"""

with open("/content/drive/MyDrive/data/data1.yaml", "w") as f:
    f.write(data_yaml_content)

print("YAML file created")


YAML file created


In [ ]:
model=YOLO('yolov8s.yaml')

result = model.train(
    data=data,
    epochs=130,
    lr0=0.001,
    imgsz=960,
    resume=True,
    patience=20,
    weight_decay=0.0005,
    save_period=10,
)


trained_model_path = os.path.join(result.save_dir, "weights", "best.pt")

destination_path = "/content/drive/MyDrive/Model/YOLO_Trained_Model.pt"
os.makedirs(os.path.dirname(destination_path), exist_ok=True)


shutil.copy(trained_model_path, destination_path)
print(f"Trained model has been saved to: {destination_path}")


Ultralytics 8.3.54 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (NVIDIA A100-SXM4-40GB, 40514MiB)
engine/trainer: task=detect, mode=train, model=yolov8s.yaml, data=/content/drive/MyDrive/data/data1.yaml, epochs=130, time=None, patience=20, batch=16, imgsz=960, save=True, save_period=10, cache=False, device=None, workers=8, project=None, name=train32, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=None, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True

train: Scanning /content/drive/MyDrive/data/food.cache... 7888 images, 2482 backgrounds, 0 corrupt: 100%|██████████| 8615/8615 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))



val: Scanning /content/drive/MyDrive/data/food.cache... 7888 images, 2482 backgrounds, 0 corrupt: 100%|██████████| 8615/8615 [00:00<?, ?it/s]


Plotting labels to runs/detect/train32/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.001' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: SGD(lr=0.01, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 960 train, 960 val
Using 8 dataloader workers
Logging results to runs/detect/train32
Starting training for 130 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

                   all       8615       9798      0.835      0.834      0.887      0.696

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


    130/130      8.38G     0.8269     0.8048      1.518         10        960: 100%|██████████| 539/539 [01:24<00:00,  6.41it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 270/270 [00:48<00:00,  5.57it/s]


                   all       8615       9798      0.837      0.834      0.888      0.697

130 epochs completed in 4.901 hours.
Optimizer stripped from runs/detect/train32/weights/last.pt, 22.6MB
Optimizer stripped from runs/detect/train32/weights/best.pt, 22.6MB

Validating runs/detect/train32/weights/best.pt...
Ultralytics 8.3.54 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (NVIDIA A100-SXM4-40GB, 40514MiB)
YOLOv8s summary (fused): 168 layers, 11,133,324 parameters, 0 gradients, 28.5 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 270/270 [00:49<00:00,  5.43it/s]


                   all       8615       9798      0.837      0.834      0.888      0.697
                 Pizza        299        534      0.892      0.909       0.95      0.748
                Mansaf        353        404       0.88      0.866      0.933      0.718
                Mjadra        461        482      0.844      0.739      0.887      0.654
               Maqluba        283        287      0.909      0.948      0.966      0.803
               Fasolia        396        426      0.666      0.898      0.759      0.687
                Awameh        359       1263       0.75      0.509      0.665      0.459
                 Salad        348        375      0.921       0.96      0.972      0.797
              Molokhia        295        465      0.942      0.805      0.926      0.698
              Shawarma        283        709      0.781      0.907      0.926      0.688
              Macaroni        273        317      0.923      0.915      0.953      0.814
                Burge

In [4]:
model_path="drive/MyDrive/final_version.pt"

##6- Evaluation & Visualization :


*   We use [mAP50 and mAP50-95](https://jonathan-hui.medium.com/map-mean-average-precision-for-object-detection-45c121a31173), along with precision and recall, because they are standard metrics for object detection tasks, evaluating both localization and classification accuracy. Unlike classification reports, these metrics specifically account for the accuracy of bounding box predictions in addition to class identification, which is crucial for object detection performance.
*   Test the model on new images to see if it is good or not.






In [ ]:
cache_file = "drive/MyDrive/data/test.cache"
if os.path.exists(cache_file):
    os.remove(cache_file)
    print(f"Removed existing cache file: {cache_file}")

Removed existing cache file: drive/MyDrive/data/test.cache


In [ ]:
model = YOLO(model_path)

metrics = model.val(data=data, cache=False)
mAP50=metrics.results_dict.get('metrics/mAP50(B)', 'Not Available')
mAP50_95=metrics.results_dict.get('metrics/mAP50-95(B)', 'Not Available')
precision = metrics.results_dict.get('metrics/precision(B)', 0)
recall = metrics.results_dict.get('metrics/recall(B)', 0)

print("mAP50:",mAP50)
print("mAP50-95:", mAP50_95)
print("Precision:", precision)
print("Recall:", recall)

if precision != 0 and recall != 0:
    f1_score = 2 * (precision * recall) / (precision + recall)
    print("Calculated F1 Score:", f1_score)
else:
    print("Precision or Recall is zero, F1 Score cannot be calculated.")

Ultralytics 8.3.196 🚀 Python-3.12.11 torch-2.8.0+cu126 CPU (Intel Xeon CPU @ 2.20GHz)
YOLOv8s summary (fused): 72 layers, 11,133,324 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.8±0.3 ms, read: 66.3±30.3 MB/s, size: 195.2 KB)
val: Scanning /content/drive/.shortcut-targets-by-id/1hhZuflxfk9fMUwkEitPFSMG1o1zfxCqT/data/test.cache... 656 images, 304 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 825/825 710.7Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 52/52 0.0it/s 26:39
                   all        825        786      0.742      0.835      0.809      0.647
                 Pizza         21         31      0.853      0.935      0.939      0.751
                Mansaf         25         27      0.724      0.874      0.785      0.601
                Mjadra         31         32      0.687      0.958      0.839      0.647
               Maqluba         32         33      0.959          1      0.98

In [5]:
model = YOLO(model_path)
if model:
    prediction_directory = "/content/drive/MyDrive/Colab Notebooks/Gradation project/Data_prediction"
    if not os.path.exists(prediction_directory):
        print(f"Error: Directory not found at {prediction_directory}")
    else:
        image_files = [f for f in os.listdir(prediction_directory) if f.lower().endswith(('.jpeg', '.jpg', '.png', '.gif', '.bmp', '.tiff', '.tif', '.webp', '.heic', '.heif', '.cr2', '.cr3', '.nef', '.nrw', '.arw', '.raf', '.orf', '.rw2', '.dng', '.svg', '.psd','.ico'))]

        if not image_files:
            print(f"No image files found in {prediction_directory}")
        else:
            print(f"Found {len(image_files)} image files in {prediction_directory}")
            for image_file in image_files:
                img_path = os.path.join(prediction_directory, image_file)

                img = cv2.imread(img_path)
                if img is None:
                    print(f"Error: Failed to load image {img_path}. Skipping.")
                    continue
                img_resized = cv2.resize(img, (960, 960))
                results = model.predict(img_resized, conf=0.45)
                annotated_img = results[0].plot()
                plt.figure(figsize=(7, 7))
                plt.imshow(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB))
                plt.axis('off')
                plt.show()

                if results[0].boxes is not None and len(results[0].boxes) > 0:
                    for box in results[0].boxes:
                        class_id = int(box.cls)
                        label = model.names[class_id]
                        confidence = box.conf.item()
                        print(f"- {label}: {confidence:.2f}")
                else:
                    print("No detections found for this image.")

Output hidden; open in https://colab.research.google.com to view.

##The insight of above cell :
The model can recognize the meal from via image and drow good boxes around that meal , so we can deploy the model for application and share it

#Why I don't use augmentation?

*   I can use argumentation as like rotate ( 90 ,180 ,270 ) , random clip and flip ( horizontal , vertical ) , but each train take 2 subscription ( Colab Pro : 100  compute units per ) and take +10 hour , so it is very slow and expensive ( it take from me 100 $ to train ) .
*   The free subscription can not do that , it stop in 15 and very very slow , so if i have good GPU or more compute units per , I will enhance the model and use augmentation on data

#Why the evaluating each class using metrics not very good ?


*   The average quality of images is bad ( 500 , 500 ) and a lot of image's shape ( 240 , 240) , so that are negative effect to model .
*   The source of images from google , social media and from family